### Problem (train_bpe_tinystories): BPE Training on TinyStories(2 points)
### 问题 (train_bpe_tinystories): TinyStories上的BPE训练(2分)
(a) Train a byte-level BPE tokenizer on the TinyStories dataset, using a maximum vocabulary size of 10,000. Make sure to add the TinyStories <|endoftext|> special token to the vocabulary. Serialize the resulting vocabulary and merges to disk for further inspection. How many hours and memory did training take? What is the longest token in the vocabulary? Does it make sense?  Resource requirements: ≤ 30 minutes (no GPUs), ≤ 30GB RAM  Hint You should be able to get under 2 minutes for BPE training using multiprocessing during pretokenization and the following two facts:  

(a) The <|endoftext|> token delimits documents in the data files.  

(b) The <|endoftext|> token is handled as a special case before the BPE merges are applied.  Deliverable: A one-to-two sentence response.

(a) 在TinyStories数据集上训练一个字节级BPE分词器，使用最大词汇量10,000。确保将TinyStories的
  <|endoftext|>特殊token添加到词汇表中。将生成的词汇表和合并操作序列化到磁盘以供进一步检查。
  训练耗时多少小时，消耗了多少内存？词汇表中最长的token是什么？它合理吗？
  资源要求：≤30分钟（无GPU），≤ 30GB内存

  提示：你应该能够通过在预分词期间使用多进程处理以及以下两个事实，将BPE训练时间控制在2分钟以
  内：

(a) <|endoftext|> token用于分隔数据文件中的文档。
(b) <|endoftext|> token在应用BPE合并之前作为特殊情况处理。

交付物：一到两句话的回答。

In [2]:
from cs336_basics.train_bpe import train_bpe
from tests.common import gpt2_bytes_to_unicode
from pathlib import Path
import json, time, psutil, os

# Auto-detect input path; fall back to sample fixture if full dataset is missing
candidates = [
    Path('/Users/yuanquan/code_project/stanfordCS336/assignment1-basics/data/TinyStoriesV2-GPT4-valid.txt'),
    Path('data/TinyStoriesV2-GPT4-train.jsonl'),
]
fallback = Path('tests/fixtures/tinystories_sample_5M.txt')
input_path = next((p for p in candidates if p.exists()), fallback)
print('Using input_path =', input_path)

vocab_size = 10_000
special_tokens = ['<|endoftext|>']
num_workers = os.cpu_count() or 8  # adjust as needed


Using input_path = tests/fixtures/tinystories_sample_5M.txt


In [8]:
proc = psutil.Process()
rss_before = proc.memory_info().rss
t0 = time.time()

vocab, merges = train_bpe(
    input_path=input_path,
    vocab_size=vocab_size,
    special_tokens=special_tokens,
    num_workers=num_workers,
)

t1 = time.time()
rss_after = proc.memory_info().rss
elapsed_s = t1 - t0
peak_gb = max(rss_before, rss_after) / (1024**3)
print(f'Elapsed: {elapsed_s:.2f}s, Approx RSS: {peak_gb:.2f} GB')


Elapsed: 81.21s, Approx RSS: 0.08 GB


In [10]:
# Save vocab (int->readable str) and merges (readable pairs)
b2u = gpt2_bytes_to_unicode()

def to_readable(bs: bytes) -> str:
    return ''.join(b2u[b] for b in bs)

vocab_json = {int(i): to_readable(tok) for i, tok in vocab.items()}
from pathlib import Path
out_dir=Path('outputs'); out_dir.mkdir(exist_ok=True)
with open(out_dir/'tinystories_vocab_10k.json', 'w', encoding='utf-8') as f:
    json.dump(vocab_json, f, ensure_ascii=False, indent=2)

with open(out_dir/'tinystories_merges_10k.txt', 'w', encoding='utf-8') as f:
    for left, right in merges:
        f.write(f"{to_readable(left)} {to_readable(right)}\n")

print('Saved tinystories_vocab_10k.json and tinystories_merges_10k.txt')


Saved tinystories_vocab_10k.json and tinystories_merges_10k.txt


In [11]:
# Inspect longest token
longest = max(vocab.values(), key=len)
longest_readable = to_readable(longest)
print('Longest token length (bytes):', len(longest))
print('Longest token (readable):', repr(longest_readable))

# Deliverable sentence (1–2 sentences)
hours = elapsed_s / 3600
print(
    f"在 TinyStories 上训练字节级 BPE（vocab_size=10k，含 <|endoftext|>），"
    f"耗时约 {hours:.2f} 小时（{elapsed_s/60:.1f} 分钟），"
    f"峰值内存约 {peak_gb:.2f} GB；"
    f"最长 token {len(longest)} 字节，表现为 {repr(longest_readable)}。"
)


Longest token length (bytes): 15
Longest token (readable): 'Ġaccomplishment'
在 TinyStories 上训练字节级 BPE（vocab_size=10k，含 <|endoftext|>），耗时约 0.02 小时（1.4 分钟），峰值内存约 0.08 GB；最长 token 15 字节，表现为 'Ġaccomplishment'。
